# Build the eFlesh touch pad

You are going to generate the soft pad that sits inside the armband, and export
it as a printable STL.

**How the pad works.** It is a lattice -- mostly air -- with four magnets buried
inside it. Press the pad, the lattice squashes, the magnets move a fraction of a
millimetre, and the sensors underneath read that movement. A stiff pad reads
nothing, so most of the numbers you fill in are really about how soft it is.

**What you will do.** Eight steps, about 35 minutes. Each one has blanks marked
`None`. Everything else is written for you, including all the plotting.

| Step | What it does | Minutes |
|---|---|---|
| 1 | Make the outline box | 4 |
| 2 | Choose the cell size and softness | 6 |
| 3 | Fill the box with lattice | 5 |
| 4 | Tidy up the mesh | 3 |
| 5 | Build one magnet pocket | 6 |
| 6 | Put four pockets in the lattice | 4 |
| 7 | Add the base plate | 4 |
| 8 | Check it, then export | 3 |

This follows `eFlesh/generate_lattice_v10_5.py`, the script that made the real
pad. Same cell pattern, same cell size, same magnet pockets. The finished part
comes out within about 2% of the real one by volume.

One difference: eFlesh shapes its lattice with a C++ program that only builds on
Linux. Step 3 does the same job in Python instead. It uses eFlesh's real cell
pattern and real strut thickness, so the result is close, but it is not the same
program. Step 3 explains exactly what differs.


## Setup

Install once, from the project folder.

Windows:
```powershell
.\.venv\Scripts\python.exe -m pip install -r .\notebooks\week2\requirements.txt
.\.venv\Scripts\python.exe -m jupyter lab
```

Mac:
```bash
./.venv/bin/python -m pip install -r notebooks/week2/requirements.txt
./.venv/bin/python -m jupyter lab
```

Run the cell below. It should print two paths. Your files are saved to
`notebooks/week2/output/`.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import trimesh


def find_week2_dir(start=Path.cwd()):
    for folder in (start.resolve(), *start.resolve().parents):
        if (folder / "week2_lattice.py").exists():
            return folder
        if (folder / "notebooks" / "week2" / "week2_lattice.py").exists():
            return folder / "notebooks" / "week2"
    raise FileNotFoundError("Open Jupyter from inside the project folder.")


sys.path.insert(0, str(find_week2_dir()))

from week2_lattice import (
    describe_mesh,
    drop_debris,
    find_repo_root,
    inflate_lattice,
    load_material_model,
    output_dir,
    plot_section,
    read_cell_pattern,
    slicer_notes,
)

REPO = find_repo_root()
OUT = output_dir()
print("project:", REPO)
print("output: ", OUT)


## Step 1 — Make the outline box

The lattice generator fills whatever shape you give it. So we start with a plain
box: **30 mm wide, 30 mm deep, 13 mm tall**. That is the pad, before the base
plate goes on in step 7.

The box does not sit at `(0, 0)`. We slide it **1 mm** across in X and Y.

**Why the 1 mm shift?** The lattice pattern repeats on a fixed grid in space,
like floor tiles. Your part sits on top of those tiles wherever it happens to
land. Without the shift, the four magnets end up in four different spots on the
tile pattern, so each one gets a different amount of lattice holding it up — and
the four sensors stop matching. With the shift, the pad is centred on a tile
corner and all four magnets sit identically. Step 8 checks this.

**Fill in:** the three box dimensions, the shift, and the move.

**You should see:** the box spans `(1, 1, 0)` to `(31, 31, 13)`.

**Hint.** `trimesh.creation.box` builds the box around `(0, 0, 0)`, so half of it
is in negative space. Each entry below moves it fully positive, then adds the
shift. Z gets no shift.

```python
move = [BOX_X / 2 + ???, BOX_Y / 2 + ???, HEIGHT / 2]
```


In [ ]:
# TODO: fill in the four values, in millimetres.
BOX_X = None
BOX_Y = None
HEIGHT = None
SHIFT = None

assert None not in (BOX_X, BOX_Y, HEIGHT, SHIFT), "Fill in all four."

box = trimesh.creation.box(extents=[BOX_X, BOX_Y, HEIGHT])

# TODO: fill in the move. See the hint.
move = None

assert move is not None, "A list of three numbers."
box.apply_translation(move)

np.testing.assert_allclose(box.bounds, [[1, 1, 0], [31, 31, 13]], atol=1e-9)
middle = (box.bounds[0] + box.bounds[1]) / 2
print("box spans", box.bounds[0], "to", box.bounds[1])
print("centre of the pad is now at x =", middle[0], " y =", middle[1])
print("Step 1 done.")


## Step 2 — Choose the cell size and the softness

The lattice is one small shape repeated over and over. One copy is a **cell**.
We use **8 mm** cells, the same as the real pad.

Now the interesting part. You do **not** pick how thick the struts are. You pick
how soft you want the material to be, and eFlesh works backwards to the strut
thickness for you. That conversion is a lookup table built from real test
results, and it runs here in Python.

The real pad asks for different softness at different heights:

```python
SOFTNESS = {-1: 0.0001, 0: 0.0001, 1: 0.0002}
```

Smaller number means softer. The bottom two layers are as soft as the table
goes; the top layer is twice as stiff. That is on purpose: only the lattice
*under* a magnet can push it toward the sensor when you press, so the softness
is spent down low and the top is left firmer.

We also need to know which cells the box covers. The grid is fixed in space, so
you divide the box corners by the cell size.

**Fill in:** the cell size, the softness table, and the two cell ranges.

**You should see:** cells `[0, 0, -1]` through `[3, 3, 1]`, which is 48 cells.

**Hint.** `low` and `high` are the box corners from step 1. Keep them as arrays.
Note that only the first line subtracts 1 — that extra cell makes the pattern
start just outside the box, so no strut gets cut off short at the edge.

```python
first_cell = np.ceil(??? / CELL).astype(int) - 1
last_cell  = np.floor(??? / CELL).astype(int)
```


In [ ]:
# TODO: fill in the cell size (mm) and the softness table.
CELL = None
SOFTNESS = None

assert None not in (CELL, SOFTNESS), "Fill in both."

low, high = box.bounds

# TODO: fill in the first and last cell the box touches.
first_cell = None
last_cell = None

assert first_cell is not None and last_cell is not None, "Divide the corners by CELL."
n_cells = int(np.prod(last_cell - first_cell + 1))

assert list(first_cell) == [0, 0, -1], f"expected [0, 0, -1], got {list(first_cell)}"
assert list(last_cell) == [3, 3, 1], f"expected [3, 3, 1], got {list(last_cell)}"
assert n_cells == 48, f"expected 48 cells, got {n_cells}"
print("cells across x, y, z:", last_cell - first_cell + 1)
print("total cells:", n_cells)


Now run the lookup. It turns each softness value into strut thicknesses.


In [ ]:
material = load_material_model(REPO)

for layer, softness in sorted(SOFTNESS.items()):
    thicknesses = [t * CELL for t in material.thicknesses(softness)]
    thinnest = min(thicknesses)
    printable = "ok" if thinnest >= 0.4 else "TOO THIN for a 0.4 mm nozzle"
    print(f"layer {layer:+d}  softness {softness:<8} thinnest strut {thinnest:.3f} mm   {printable}")

print()
print("The real pad's notes quote 0.628 mm as its thinnest strut. That matches.")


## Step 3 — Fill the box with lattice

Now we turn those cells into actual 3D shape.

Two of the three ingredients are real eFlesh data, sitting in this project:

- **The cell shape.** `pattern0646.wire` is a list of 62 points and 72 lines —
  the skeleton of one cell. It gets copied into all 48 cells and the lines are
  thickened into round struts.
- **The strut thickness.** It comes from the lookup you just ran.

**What is different from the real pad.** eFlesh gives each strut one of four
thicknesses and rounds off the joints where struts meet. Both of those live in
the C++ program. Here every strut gets the same thickness — the average of
eFlesh's four. That lands the finished part within about 2% of the real one by
volume, but the struts are more uniform than the real thing.

**Fill in:** the box size, where the box starts, the cell size, and the strut
radius.

**You should see:** about 600,000 triangles, filling the same space as your box.
Takes a few seconds.

**Hint.** `start` is the box's lowest corner — that is where the 1 mm shift
lives. Radius is half the thickness.

```python
start = (???, ???, 0.0)
strut_radius = THICKNESS / 2
```


In [ ]:
points, lines = read_cell_pattern(REPO)
print(f"one cell: {len(points)} points, {len(lines)} lines")
print(f"across {n_cells} cells: {len(lines) * n_cells:,} struts")

# The average of eFlesh's four strut thicknesses, from the step 2 lookup.
THICKNESS = float(np.mean([t * CELL for t in material.thicknesses(SOFTNESS[0])]))
print(f"strut thickness: {THICKNESS:.3f} mm")
print()

# TODO: fill in the four values. Takes a few seconds to run.
lattice = inflate_lattice(
    REPO,
    size=None,
    start=None,
    cell_size=None,
    strut_radius=None,
    first_cell=first_cell,
    last_cell=last_cell,
)

describe_mesh(lattice, "lattice")

assert lattice.is_watertight, "The mesh has holes in it."
np.testing.assert_allclose(lattice.bounds, box.bounds, atol=0.05)
solid = 100 * lattice.volume / (BOX_X * BOX_Y * HEIGHT)
assert 11.0 <= solid <= 15.0, "Check THICKNESS and CELL."
print()
print(f"{solid:.0f}% solid, {100 - solid:.0f}% air. That is what makes it squashy.")


## Step 4 — Tidy up the mesh

Building a surface this way leaves small defects: duplicate points where cells
meet, and a few tiny loose fragments at the edges of the box. They cause
confusing failures later, so we clear them now.

The last call, `drop_debris`, deletes small disconnected pieces. Later, once the
magnet pockets exist, it needs a list of places to leave alone — a pocket wall
counts as a small disconnected piece, and an earlier version of eFlesh deleted
all four of them by accident. For now there are no pockets, so nothing to
protect.

**Fill in:** the three clean-up calls, by name.

**You should see:** the same bounds, still watertight.

**Hint.** All three are methods on `lattice` that take no arguments, in this
order: merge duplicate points, delete points no longer used, then make every
surface face outward.

```python
for name in ("merge_vertices", ???, ???):
```


In [ ]:
before = len(lattice.faces)
lattice.update_faces(lattice.nondegenerate_faces())

# TODO: fill in the three method names, as strings. See the hint.
for name in (None, None, None):
    assert name is not None, "Three names, in order."
    getattr(lattice, name)()

lattice = drop_debris(lattice)

assert lattice.is_watertight, "Clean-up left holes."
np.testing.assert_allclose(lattice.bounds, box.bounds, atol=0.05)
print(f"{before:,} triangles -> {len(lattice.faces):,} after tidying")


## Step 5 — Build one magnet pocket

Four magnets go inside the pad, **4.8 mm across and 1.6 mm thick**, sitting at a
height of **6.5 mm**.

You cannot just drill a hole. The lattice is nearly 90% air, so a hole removes
almost nothing and the magnet would drop straight through the gaps. Instead we
drop in a **solid plug** and hollow the pocket out of that.

Four measurements shape the plug, and each was settled by actually printing it:

- **0.25 mm extra width** so the magnet drops in instead of jamming.
- **0.8 mm of solid underneath.** Thinner is better for sensing, because the
  lattice below the magnet is the only part that can carry it toward the sensor.
  But the printer has to bridge this gap in mid-air: 0.4 mm sagged, 0.6 mm was
  still rough, 0.8 mm came out flat.
- **1.0 mm of solid on top**, printed after you drop the magnet in. That is what
  holds it in.
- **0.1 mm extra depth** so the magnet never sticks up proud.

**Fill in:** the pocket width, and where the plug starts and ends.

**You should see:** plug from 4.0 to 7.5 mm, pocket from 4.8 to 6.5 mm.

**Hint.** The plug is the pocket plus solid below and above it.

```python
POCKET_RADIUS = MAGNET_RADIUS + ???
PLUG_BOTTOM = POCKET_BOTTOM - ???
PLUG_TOP = POCKET_TOP + ???
```


In [ ]:
# Given: the real pad's magnet measurements, in millimetres.
MAGNET_RADIUS = 4.8 / 2
MAGNET_THICK = 1.6
MAGNET_HEIGHT = 6.5     # where the magnet sits
EXTRA_WIDTH = 0.25      # so it drops in
EXTRA_DEPTH = 0.10      # so it never sits proud
SOLID_BELOW = 0.8       # floor under the magnet
SOLID_ABOVE = 1.0       # roof over the magnet
PLUG_RADIUS = MAGNET_RADIUS + 1.0

POCKET_DEPTH = MAGNET_THICK + EXTRA_DEPTH
POCKET_TOP = MAGNET_HEIGHT
POCKET_BOTTOM = POCKET_TOP - POCKET_DEPTH

# TODO: fill in the three values.
POCKET_RADIUS = None
PLUG_BOTTOM = None
PLUG_TOP = None

assert None not in (POCKET_RADIUS, PLUG_BOTTOM, PLUG_TOP), "Fill in all three."
PLUG_HEIGHT = PLUG_TOP - PLUG_BOTTOM

assert abs(POCKET_RADIUS - 2.65) < 1e-9, f"expected 2.65, got {POCKET_RADIUS}"
assert abs(PLUG_BOTTOM - 4.0) < 1e-9, f"expected 4.0, got {PLUG_BOTTOM}"
assert abs(PLUG_TOP - 7.5) < 1e-9, f"expected 7.5, got {PLUG_TOP}"
print(f"plug:   {PLUG_BOTTOM} to {PLUG_TOP} mm tall, radius {PLUG_RADIUS} mm")
print(f"pocket: {POCKET_BOTTOM} to {POCKET_TOP} mm tall, radius {POCKET_RADIUS} mm")


Now make the plug: a solid cylinder with the pocket cut out of it.

**Hint.** Both cylinders are built around `(0, 0, 0)`, so each needs moving up to
its own middle. The middle of something is halfway between its bottom and top.

```python
plug.apply_translation([0.0, 0.0, (PLUG_BOTTOM + PLUG_TOP) / 2])
...
pocket_cut = plug.difference(pocket, engine="manifold")
```


In [ ]:
def make_plug():
    plug = trimesh.creation.cylinder(radius=PLUG_RADIUS, height=PLUG_HEIGHT, sections=64)
    pocket = trimesh.creation.cylinder(radius=POCKET_RADIUS, height=POCKET_DEPTH + 0.02, sections=64)

    # TODO: fill in the middle height of each cylinder.
    plug_middle = None
    pocket_middle = None

    assert None not in (plug_middle, pocket_middle), "Halfway between bottom and top."
    plug.apply_translation([0.0, 0.0, plug_middle])
    pocket.apply_translation([0.0, 0.0, pocket_middle])

    # TODO: cut the pocket out of the plug.
    result = None

    assert result is not None, 'Use plug.difference(pocket, engine="manifold").'
    return result


one = make_plug()
np.testing.assert_allclose(one.bounds[:, 2], [PLUG_BOTTOM, PLUG_TOP], atol=1e-6)
assert one.is_watertight, "The cut left holes."
assert one.volume < np.pi * PLUG_RADIUS ** 2 * PLUG_HEIGHT - 10, "Nothing was cut out."
print(f"one plug: {len(one.faces)} triangles, {one.volume:.1f} mm3 of solid")


## Step 6 — Put four pockets in the lattice

The four magnets sit in a square, 12 mm apart: `(9, 9)`, `(21, 9)`, `(9, 21)`,
`(21, 21)`.

For each one we do two things:

1. **Clear a hole** in the lattice, so no struts are in the way.
2. **Drop the plug in** and join it to the lattice.

The hole is cut slightly **narrower** than the plug, by 0.15 mm. That way the
plug overlaps the surrounding lattice and fuses to it. If the hole were exactly
plug-sized the two would only touch, which both the 3D software and the slicer
handle badly.

Remember the lattice is shifted 1 mm, but these magnet positions are not. You
need to add the shift.

**Fill in:** the shifted position, the hole radius, and the join.

**You should see:** four pockets placed, still watertight. This is the slowest
cell, around 12 seconds.

**Hint.**

```python
x, y = x0 + ???, y0 + ???
hole_radius = PLUG_RADIUS - ???
lattice = lattice.union(???, engine="manifold")
```


In [ ]:
MAGNET_SPOTS = ((9.0, 9.0), (21.0, 9.0), (9.0, 21.0), (21.0, 21.0))
POLES = "N S N S"
OVERLAP = 0.15

for number, (x0, y0) in enumerate(MAGNET_SPOTS, start=1):
    # TODO: shift the position to match the lattice.
    x, y = None, None

    assert None not in (x, y), "The lattice moved 1 mm; these positions have not."
    hole_height = PLUG_HEIGHT + 0.5

    # TODO: fill in the hole radius.
    hole_radius = None

    assert hole_radius is not None, "Slightly narrower than PLUG_RADIUS."
    hole = trimesh.creation.cylinder(radius=hole_radius, height=hole_height, sections=64)
    hole.apply_translation([x, y, PLUG_BOTTOM + hole_height / 2])
    lattice = lattice.difference(hole, engine="manifold")

    plug = make_plug()
    plug.apply_translation([x, y, 0.0])

    # TODO: join the plug to the lattice.
    lattice = None

    assert lattice is not None, "Union the plug in."
    print(f"pocket {number} placed at ({x0}, {y0})")

shifted_spots = tuple((x + SHIFT, y + SHIFT) for x, y in MAGNET_SPOTS)
lattice = drop_debris(lattice, keep_near=shifted_spots)

print()
describe_mesh(lattice, "lattice with pockets")
assert lattice.is_watertight, "A cut or join left holes."


## Step 7 — Add the base plate

The pad needs a flat **1 mm solid floor**. The sensor board mounts against it,
and it gives the print something to stick to.

Two moves: slide the lattice back off its 1 mm shift, and lift it up by 1 mm so
the plate fits underneath. Then join the plate on.

Notice the plate gets `subdivide()` called on it three times first. A box is only
12 triangles, and joining 12 triangles to a 600,000-triangle lattice is exactly
where an earlier version of eFlesh lost its base plate completely — the output
came out with no floor and nothing warned about it. Splitting the plate into 768
triangles first gives the join enough to work with.

**Fill in:** the move back, the plate position, and the join.

**You should see:** a part 30 × 30 × 14 mm, watertight.

**Hint.** The move undoes the shift in X and Y, and lifts by the plate
thickness. The plate's centre is half its own height off the ground.

```python
move_back = [-SHIFT, ???, ???]
plate_centre = [BOX_X / 2, BOX_Y / 2, ???]
```


In [ ]:
BASE = 1.0
BITE = 0.02     # makes the plate 0.02 mm taller so it bites into the lattice

pad = lattice.copy()

# TODO: fill in both.
move_back = None
plate_centre = None

assert None not in (move_back, plate_centre), "Fill in both."
pad.apply_translation(move_back)

plate = trimesh.creation.box(extents=(BOX_X, BOX_Y, BASE + BITE))
plate.apply_translation(plate_centre)
for _ in range(3):
    plate = plate.subdivide()

# TODO: join the lattice onto the plate.
part = None

assert part is not None, "Union them."
part = drop_debris(part, keep_near=MAGNET_SPOTS)
part.fix_normals()

describe_mesh(part, "finished part")
assert part.is_watertight, "The join left holes."
np.testing.assert_allclose(part.bounds, [[0, 0, 0], [30, 30, 14]], atol=0.05)

plot_section(part, y=9.0, title="Cut through the pad at y = 9 mm")


In that picture you can see the whole design: the solid floor along the bottom,
the open lattice above it, and two magnet pockets as rectangles with solid
material under and over them.


## Step 8 — Check it, then export

Two checks before exporting. Both exist because both have failed before.

1. **Are the pockets actually hollow?** Once, a tidy-up step sealed all four
   solid, and every other check still passed. The part printed with nowhere to
   put the magnets.
2. **Do all four pockets look the same?** Count the triangles around each one.
   They should match exactly. If they do not, the 1 mm shift from step 1 is
   wrong and your four sensors will not match.

**Fill in:** the pocket height and the distance test.

**You should see:** four hollow pockets, all with the same triangle count.

**Hint.** Everything moved up by `BASE` in step 7.

```python
check_height = (POCKET_BOTTOM + POCKET_TOP) / 2 + ???
distance = np.sqrt((centres[:, 0] - x) ** 2 + (centres[:, 1] - y) ** 2)
```


In [ ]:
# TODO: the middle height of a pocket, in the finished part.
check_height = None

assert check_height is not None, "The part was lifted by the base plate."
probes = np.array([[x, y, check_height] for x, y in MAGNET_SPOTS])
for (x, y), is_solid in zip(MAGNET_SPOTS, part.contains(probes)):
    assert not is_solid, f"pocket at ({x}, {y}) is solid -- it got sealed up"
print(f"all 4 pockets are hollow at height {check_height:.2f} mm")

centres = part.triangles_center
inside = (centres[:, 2] > PLUG_BOTTOM + BASE + 0.05) & (centres[:, 2] < PLUG_TOP + BASE - 0.05)
counts = []
for x, y in MAGNET_SPOTS:
    # TODO: how far is each triangle from (x, y), ignoring height?
    distance = None

    assert distance is not None, "Use the first two columns of centres."
    counts.append(int(np.count_nonzero(inside & (distance < PLUG_RADIUS - OVERLAP - 0.05))))

for (x, y), count, pole in zip(MAGNET_SPOTS, counts, POLES.split()):
    print(f"  pocket ({x:4.1f}, {y:4.1f})  pole {pole}  {count} triangles")

assert len(set(counts)) == 1, f"pockets differ: {counts} -- check SHIFT in step 1"
print()
print("All four match. The 1 mm shift did its job.")


Last step: save it.

**Fill in:** the file name.


In [ ]:
# TODO: name the file.
NAME = None

assert NAME is not None, "Give it a name ending in .stl"
path = OUT / NAME
part.export(path)

check = trimesh.load(path, force="mesh")
print("saved:", path)
print(f"  {path.stat().st_size / 1e6:.1f} MB, {len(check.faces):,} triangles")
print(f"  size {np.round(check.bounding_box.extents, 2)} mm")
assert path.suffix == ".stl"
np.testing.assert_allclose(check.bounding_box.extents, [30, 30, 14], atol=0.05)
print()
print("Done. This file is ready to slice and print.")


## Printing it

The important line is the pause. The printer stops partway, you drop the four
magnets in, and it carries on and seals them under 1 mm of solid.

The poles alternate N, S, N, S. If all four faced the same way their fields
would blur into one and no sensor could tell which magnet moved.


In [ ]:
print(slicer_notes(MAGNET_HEIGHT, BASE, POLES, MAGNET_SPOTS))


## How close is it?

Compare what you made against the real pad.


In [ ]:
real = trimesh.load(REPO / "eFlesh/output/lattice_v10_5.stl", force="mesh")
print(f"{'':20}{'yours':>12}{'real pad':>12}")
print(f"{'triangles':20}{len(check.faces):>12,}{len(real.faces):>12,}")
print(f"{'solid (mm3)':20}{check.volume:>12,.0f}{real.volume:>12,.0f}")
print(f"{'width (mm)':20}{check.bounding_box.extents[0]:>12.2f}{real.bounding_box.extents[0]:>12.2f}")
print(f"{'height (mm)':20}{check.bounding_box.extents[2]:>12.2f}{real.bounding_box.extents[2]:>12.2f}")
print()
print(f"solid volume is within {abs(100 * check.volume / real.volume - 100):.0f}% of the real pad")


## Try changing something

Change one number and run the notebook again. The whole thing takes about 25
seconds.

**Softer or firmer:**

- `SOLID_BELOW` (step 5) — the floor under each magnet. Thinner means more
  lattice can squash under the magnet, so the sensor reads more. But thinner
  also sagged when printed. 0.6 mm is the lowest that worked.
- `MAGNET_HEIGHT` (step 5) — lower means the magnet sits closer to the sensor,
  which helps a lot. But it leaves less lattice underneath to squash.
- `SOFTNESS` (step 2) — ask for a softer material and the struts get thinner.
  Watch the printable check: below 0.4 mm the nozzle cannot make them.

**Shape and layout:**

- `CELL` (step 2) — bigger cells mean fewer, longer struts.
- `MAGNET_SPOTS` (step 6) — move the magnets. Keep them away from the edges or
  the pocket walls break out.

**Break it on purpose:** set `SHIFT = 0` in step 1 and watch step 8 fail. That
is the check doing its job.
